## 04c: Minimum Detectable Effect Sizes

The minimum detectable effect size (MDES) at 80% power for each test, given its sample size.
Power under Benjamini-Hochberg has no closed form, so for the 28 primary tests the MDES is
bounded between the nominal (α = .05) and Bonferroni (α = .05/28) thresholds. The NCC arm
(`HC` in the code) uses α = .05 only.

- Normative deviation: MDES as Glass's Δ (one-sample t-test per group; two-sample for the
  BD − MDD contrast).
- Rest-activity phenotypes: MDES as Pearson r (Fisher z approximation).

Run after `04a` and `04b`: the MDES columns are added in place to
`results/model_results.parquet` and `results/model_results_hc_sensitivity.parquet`. Task 2
gives the MDES a BD-only phenotype analysis would have had, which is why that analysis is
fitted in MDD only (`results/mdes_stratified_posthoc.parquet`).

## Task 0: Setup

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from statsmodels.stats.power import TTestPower, TTestIndPower
from scipy.stats import norm

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *
from src.analysis import (
    ETM_TASKS, PHENOTYPES, MIN_CELL, MAIN_POOL_SIZE,
    prepare_cohort, normative_sample, fitbit_sample, phenotype_complete,
)

OVERWRITE = True
RES_OUT = "nb04c"

# ── Load upstream outputs (must run 04a then 04b before this notebook) ─────────────
results_df = load_parquet('results/model_results.parquet')
hc_df      = load_parquet('results/model_results_hc_sensitivity.parquet')

# ── Load cohort/wearable_features to rebuild the BD Fitbit sample sizes (Task 2) ──
cohort = prepare_cohort(load_parquet('cohort/cohort_master.parquet'))
wf     = load_parquet('features/wearable_features.parquet')

print(f'model_results.parquet             : {len(results_df):,} rows')
print(f'model_results_hc_sensitivity.parquet: {len(hc_df):,} rows')
print(f'OVERWRITE                          : {OVERWRITE}')


## Task 1: MDES for the main pool and the HC sensitivity arm

In [ ]:
_ps1 = TTestPower()
_ps2 = TTestIndPower()

def _mdes_glass_delta(row, alpha):
    """MDES (Glass's Delta) for a group mean or the BD − MDD contrast."""
    if row['suppressed']:
        return np.nan
    if row['group'] == 'contrast':
        n1, n2 = row['n_mdd'], row['n_bd']
        result = _ps2.solve_power(
            effect_size=None, nobs1=n1, ratio=n2 / n1,
            alpha=alpha, power=0.8, alternative='two-sided',
        )
    elif row['group'] == 'HC':
        n = int(row['n_hc'])
        result = _ps1.solve_power(
            effect_size=None, nobs=n,
            alpha=alpha, power=0.8, alternative='two-sided',
        )
    else:
        n = row['n_mdd'] if row['group'] == 'MDD' else row['n_bd']
        result = _ps1.solve_power(
            effect_size=None, nobs=n,
            alpha=alpha, power=0.8, alternative='two-sided',
        )
    return abs(float(result))

def _mdes_r(n, alpha):
    """MDES (Pearson r), two-sided, 80% power, via the Fisher z approximation."""
    if n is None or pd.isna(n) or n < 4:
        return np.nan
    z_crit = norm.ppf(1 - alpha / 2) + norm.ppf(0.8)
    return abs(float(np.tanh(z_crit / np.sqrt(n - 3))))

def _mdes_r_row(row, alpha, n_col):
    if row['suppressed']:
        return np.nan
    return _mdes_r(row[n_col], alpha)

# Normative deviation: Glass's Delta MDES at both alphas.
results_df['mdes_nominal'] = np.nan
results_df['mdes_bonf']    = np.nan
_normative_mask = results_df['analysis'] == 'Normative'
results_df.loc[_normative_mask, 'mdes_nominal'] = results_df.loc[_normative_mask].apply(
    lambda r: _mdes_glass_delta(r, alpha=0.05), axis=1
)
results_df.loc[_normative_mask, 'mdes_bonf'] = results_df.loc[_normative_mask].apply(
    lambda r: _mdes_glass_delta(r, alpha=0.05 / MAIN_POOL_SIZE), axis=1
)

# Rest-activity phenotypes (MDD): MDES as Pearson r and R^2.
results_df['mdes_r_nominal']  = np.nan
results_df['mdes_r_bonf']     = np.nan
results_df['mdes_r2_nominal'] = np.nan
results_df['mdes_r2_bonf']    = np.nan
_fitbit_mask = results_df['analysis'] == 'Fitbit'
results_df.loc[_fitbit_mask, 'mdes_r_nominal'] = results_df.loc[_fitbit_mask].apply(
    lambda r: _mdes_r_row(r, alpha=0.05, n_col='n_mdd'), axis=1
)
results_df.loc[_fitbit_mask, 'mdes_r_bonf'] = results_df.loc[_fitbit_mask].apply(
    lambda r: _mdes_r_row(r, alpha=0.05 / MAIN_POOL_SIZE, n_col='n_mdd'), axis=1
)
results_df.loc[_fitbit_mask, 'mdes_r2_nominal'] = results_df.loc[_fitbit_mask, 'mdes_r_nominal'] ** 2
results_df.loc[_fitbit_mask, 'mdes_r2_bonf']    = results_df.loc[_fitbit_mask, 'mdes_r_bonf'] ** 2

# HC arm: nominal alpha only
hc_df['mdes_r_nominal']  = hc_df.apply(lambda r: _mdes_r_row(r, alpha=0.05, n_col='n_hc'), axis=1)
hc_df['mdes_r2_nominal'] = hc_df['mdes_r_nominal'] ** 2

# Re-save in place, now with MDES columns
save_res(RES_OUT, 'main_res_mdes.csv',  OVERWRITE, results_df,  file_type='csv')
save_res(RES_OUT, 'sensitivity_hc_mdes.csv',  OVERWRITE, hc_df,  file_type='csv')
save_parquet(results_df, 'results/model_results.parquet')
save_parquet(hc_df,      'results/model_results_hc_sensitivity.parquet')

active = results_df[~results_df['suppressed'].fillna(False)]

a_active = active[active['analysis'] == 'Normative']
print("\n=== Normative deviation — MDES (Glass's Delta), nominal alpha = 0.05 ===")
print(a_active.pivot_table(index='group', columns='task', values='mdes_nominal').round(3).to_string())
print(f"\n=== Normative deviation — MDES (Glass's Delta), Bonferroni alpha = 0.05 / {MAIN_POOL_SIZE} ===")
print(a_active.pivot_table(index='group', columns='task', values='mdes_bonf').round(3).to_string())

b_active = active[active['analysis'] == 'Fitbit']
print('\n=== Rest-activity phenotypes — MDES (Pearson r), nominal alpha = 0.05 (MDD, main pool) ===')
print(b_active.pivot_table(index=['group', 'phenotype'], columns='task', values='mdes_r_nominal').round(3).to_string())
print('\n=== Rest-activity phenotypes — MDES (R^2), nominal alpha = 0.05 (MDD, main pool) ===')
print(b_active.pivot_table(index=['group', 'phenotype'], columns='task', values='mdes_r2_nominal').round(3).to_string())
print(f'\n=== Rest-activity phenotypes — MDES (Pearson r), Bonferroni alpha = 0.05 / {MAIN_POOL_SIZE} (MDD, main pool) ===')
print(b_active.pivot_table(index=['group', 'phenotype'], columns='task', values='mdes_r_bonf').round(3).to_string())
print(f'\n=== Rest-activity phenotypes — MDES (R^2), Bonferroni alpha = 0.05 / {MAIN_POOL_SIZE} (MDD, main pool) ===')
print(b_active.pivot_table(index=['group', 'phenotype'], columns='task', values='mdes_r2_bonf').round(3).to_string())

hc_active = hc_df[~hc_df['suppressed'].fillna(False)]
print("\n=== Rest-activity phenotypes, HC arm — MDES (Pearson r), nominal alpha = 0.05 (sensitivity) ===")
print(hc_active.pivot_table(index='phenotype', columns='task', values='mdes_r_nominal').round(3).to_string())
print("\n=== Rest-activity phenotypes, HC arm — MDES (R^2), nominal alpha = 0.05 (sensitivity) ===")
print(hc_active.pivot_table(index='phenotype', columns='task', values='mdes_r2_nominal').round(3).to_string())


## Task 2: Power of a BD-Only Phenotype Analysis

The BD wearable sample is built with the same criteria as the MDD sample; only its size is
needed, so no model is fitted. Nominal α only.

In [ ]:
bd_rows = []
for task in ETM_TASKS:
    ab_bd = normative_sample(cohort, task, 'BD')
    c_bd  = fitbit_sample(ab_bd, task, wf, 'BD')

    for pheno in PHENOTYPES:
        n_bd = len(phenotype_complete(c_bd, pheno))
        # Suppressed below 20 participants
        suppressed = n_bd < MIN_CELL
        mdes_r  = _mdes_r(n_bd, 0.05) if not suppressed else np.nan
        mdes_r2 = mdes_r ** 2 if not pd.isna(mdes_r) else np.nan
        bd_rows.append({
            'task': task, 'phenotype': pheno, 'n_bd': n_bd, 'suppressed': suppressed,
            'mdes_r_nominal':  mdes_r,
            'mdes_r2_nominal': mdes_r2,
        })

bd_df = pd.DataFrame(bd_rows)

print('=== Post-hoc power: BD-only cognition-wearable MDES (D ~ P_z), nominal alpha = 0.05 ===')
print('Context for fitting the wearable analysis in MDD only: the MDES each BD cell could detect.')
print(bd_df.set_index(['phenotype', 'task']).to_string())

save_res(RES_OUT, 'mdes_bd_posthoc.csv',  OVERWRITE, bd_df,  file_type='csv')
save_parquet(bd_df, 'results/mdes_stratified_posthoc.parquet')
